# Práctica 11 — Detección de Billetes Falsos con una Neurona
**Machine Learning · ISTER 2026 · Ing. David Minango, PhD**

---
**Objetivo:** Entrenar un Perceptrón simple para distinguir billetes genuinos de falsificados, visualizar su frontera de decisión, y calcular a mano (con NumPy) la predicción de la neurona para un billete real usando los pesos que aprendió el modelo. Al final, comparar contra un MLP con una capa oculta.

**Contexto:** Eres el analista antifraude de un banco. Te dan 4 mediciones extraídas de la imagen de cada billete (varianza, asimetría, curtosis y entropía de una transformación wavelet) y debes construir una neurona que decida si el billete es genuino o falsificado.

> ⏱️ Duración estimada: ~60 minutos
> 🔧 Completa las celdas marcadas con **TU CÓDIGO**. Las demás solo ejecútalas.
> 💾 Guarda una copia en tu Drive: `Archivo → Guardar una copia en Drive`

## Parte 0 — Setup (Solo ejecutar)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

print("✅ Librerías cargadas")

## Parte 1 — Cargar y Escalar

El dataset **Banknote Authentication** (UCI) contiene 1.372 billetes reales, descritos por 4 características extraídas de una transformación wavelet de su imagen.

In [ ]:
# Celda 1.1 — Dado: cargar los billetes
datos = fetch_openml('banknote-authentication', version=1, as_frame=True, parser='auto')
df = datos.data.rename(columns={'V1': 'varianza', 'V2': 'asimetria', 'V3': 'curtosis', 'V4': 'entropia'})
df['clase'] = (datos.target.astype(int) == 1).astype(int)   # 1 = genuino, 0 = falsificado

print(f"Billetes en el dataset: {df.shape[0]}")
print(df['clase'].value_counts().rename({1: 'genuino', 0: 'falsificado'}))
print(df.describe().round(2))

In [ ]:
# Celda 1.2 — 🔧 TU CÓDIGO: separar y escalar
features = ['varianza', 'asimetria', 'curtosis', 'entropia']
X = df[features]
y = df['clase']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                      stratify=y, random_state=42)

scaler    = ___________          # StandardScaler
X_train_s = ___________          # fit_transform sobre X_train
X_test_s  = ___________          # transform sobre X_test (¡NUNCA fit en test!)

print("Train:", X_train_s.shape, " Test:", X_test_s.shape)

### ❓ Preguntas — Antes de modelar
1. ¿Por qué usamos `fit_transform` en train pero solo `transform` en test?
2. ¿Qué pasaría si escaláramos ANTES de dividir en train/test?

_Responde aquí:_

## Parte 2 — Entrenar el Perceptrón

In [ ]:
# Celda 2.1 — 🔧 TU CÓDIGO: entrenar y evaluar
perceptron = ___________        # Perceptron(max_iter=1000, random_state=42)
___________                     # fit sobre X_train_s, y_train

y_pred = ___________             # predict sobre X_test_s
acc    = ___________             # accuracy_score(y_test, y_pred)

print(f"Accuracy del Perceptrón: {acc:.3f}")
print("Pesos aprendidos (w):", perceptron.coef_)
print("Sesgo aprendido (b):", perceptron.intercept_)

### ❓ Preguntas — Sobre el modelo
1. ¿Cuántos pesos tiene el modelo? ¿Por qué ese número (piensa en cuántas features usamos)?
2. ¿El accuracy te sorprende? Un billete falso puede costarle dinero real a un banco: ¿un Perceptrón simple te parece suficiente para producción?

_Responde aquí:_

## Parte 3 — Visualizar la Frontera de Decisión

In [ ]:
# Celda 3.1 — Dado: reentrenar en 2D para poder graficar
# Con 4 features no se puede dibujar (necesitaríamos 4 ejes).
# Reentrenamos un perceptrón nuevo usando solo 2: varianza y asimetría.
features_2d = ['varianza', 'asimetria']
X2 = df[features_2d]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.3,
                                                          stratify=y, random_state=42)

scaler_2d  = StandardScaler()
X2_train_s = scaler_2d.fit_transform(X2_train)
X2_test_s  = scaler_2d.transform(X2_test)

perc_2d = Perceptron(max_iter=1000, random_state=42)
perc_2d.fit(X2_train_s, y2_train)
print(f"Accuracy (solo 2 features): {perc_2d.score(X2_test_s, y2_test):.3f}")

In [ ]:
# Celda 3.2 — 🔧 TU CÓDIGO: graficar puntos + frontera
# w1*x1 + w2*x2 + b = 0  →  x2 = -(w1*x1 + b) / w2
w1, w2 = perc_2d.coef_[0]
b = perc_2d.intercept_[0]

xs = np.linspace(X2_train_s[:, 0].min(), X2_train_s[:, 0].max(), 50)
ys = ___________          # despeja x2 de la fórmula de arriba

plt.figure(figsize=(7, 6))
plt.scatter(X2_train_s[:, 0], X2_train_s[:, 1], c=y2_train, cmap='RdYlGn', s=25, edgecolor='k', alpha=0.7)
plt.plot(xs, ys, 'b--', lw=2, label='frontera del perceptrón')
plt.xlabel('varianza (escalada)'); plt.ylabel('asimetría (escalada)')
plt.title('Frontera de decisión del Perceptrón')
plt.legend(); plt.tight_layout(); plt.show()

### ❓ Preguntas — Sobre la frontera
1. ¿La línea separa bien los billetes genuinos (verde) de los falsificados (rojo)?
2. ¿Ves puntos del lado equivocado de la línea? ¿Qué significa eso para ese billete en particular?

_Responde aquí:_

## Parte 4 — El Cálculo a Mano

Vamos a reproducir con NumPy, paso a paso, lo que hace `.predict()` por dentro.

In [ ]:
# Celda 4.1 — Dado: tomamos UN billete real del test set
idx = 0
billete = X_test_s[idx]
clase_real = y_test.values[idx]

print("Valores escalados del billete:", billete.round(3))
print("Pesos del perceptrón (w):", perceptron.coef_[0].round(3))
print("Sesgo (b):", round(perceptron.intercept_[0], 3))
print("Clase real:", "genuino" if clase_real == 1 else "falsificado")

In [ ]:
# Celda 4.2 — 🔧 TU CÓDIGO: calcula z y la predicción SIN usar .predict()
w = perceptron.coef_[0]
b = perceptron.intercept_[0]

z = ___________            # np.dot(w, billete) + b
prediccion = ___________   # 1 if z >= 0 else 0

print(f"z = {z:.3f}")
print(f"predicción manual = {prediccion}")

# Verifica contra el modelo real:
pred_sklearn = perceptron.predict([billete])[0]
print(f"predicción de sklearn = {pred_sklearn}")
print("✅ ¡Coinciden!" if prediccion == pred_sklearn else "❌ Revisa tu cálculo")

### ❓ Preguntas — Repite y reflexiona
1. Cambia `idx` por otros 2 valores (por ejemplo 5 y 20) y repite el cálculo. ¿Siempre coincide tu resultado manual con el de sklearn?
2. Con 4 pesos y 1 sesgo, ¿cuántas multiplicaciones y sumas hace la neurona en cada predicción? Escribe la cuenta.

_Responde aquí:_

## Parte 5 — Perceptrón vs MLP

In [ ]:
# Celda 5.1 — 🔧 TU CÓDIGO: entrenar un MLP con una capa oculta
mlp = ___________          # MLPClassifier(hidden_layer_sizes=(8,), max_iter=2000, random_state=42)
___________                # fit sobre X_train_s, y_train

acc_mlp = ___________       # accuracy_score(y_test, mlp.predict(X_test_s))

print(f"Accuracy Perceptrón simple  : {acc:.3f}")
print(f"Accuracy MLP (1 capa oculta): {acc_mlp:.3f}")

### ❓ Preguntas — Sobre la comparación
1. ¿El MLP mejora, empeora o queda prácticamente igual que el Perceptrón simple?
2. Si el dataset ya es casi separable con una línea (como viste en la Parte 3), ¿esperabas una gran diferencia? ¿Por qué sí o por qué no?

_Responde aquí:_

## ✅ Entrega

1. **Notebook completo:** todas las celdas 🔧 TU CÓDIGO resueltas y ejecutadas, preguntas ❓ respondidas en celdas Markdown. Exportar como `.ipynb`.
2. **Conclusión integradora:** resume tu proyecto en un párrafo como si se lo contaras al gerente de riesgos del banco: qué tan confiable es tu neurona para detectar billetes falsos, cómo verificaste el cálculo de la neurona a mano, y si recomendarías usar un Perceptrón simple o un MLP en producción.

_Escribe tu conclusión aquí:_